# Loading improtant libraries 

# Voting Hard and Soft

In [66]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC, SVR
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, r2_score, mean_squared_error
from sklearn.ensemble import VotingClassifier, VotingRegressor, StackingRegressor
from xgboost import XGBRegressor
from joblib import load, dump 

In [39]:
df = pd.read_csv("fast_food_cheat_meal_workout_burn.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   order_id                     8000 non-null   str    
 1   fast_food_chain              8000 non-null   str    
 2   meal_category                8000 non-null   str    
 3   item_count                   8000 non-null   int64  
 4   total_calories               8000 non-null   int64  
 5   fat_grams                    8000 non-null   int64  
 6   saturated_fat_grams          8000 non-null   int64  
 7   carb_grams                   8000 non-null   int64  
 8   sugar_grams                  8000 non-null   int64  
 9   protein_grams                8000 non-null   int64  
 10  sodium_mg                    8000 non-null   int64  
 11  consumer_gender              8000 non-null   str    
 12  consumer_age                 8000 non-null   int64  
 13  consumer_weight_kg           

In [40]:
# total_caloriesni men target sifasita oldim.
X = df.drop(columns = [
    "order_id", 
    "thermic_effect_calories",
    "net_caloric_surplus", 
    "walking_burn_time_min",
    "cycling_burn_time_min",
    "running_burn_time_min",
    "hiit_burn_time_min",
    "daily_calorie_allowance_pct",
    "workout_shock_tier"
])
y = df["total_calories"]

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42
)

categorical_features = X.select_dtypes(include = ["str"]).columns.tolist()
numerical_features = [col for col in X.columns if col not in categorical_features]
print(categorical_features)

['fast_food_chain', 'meal_category', 'consumer_gender', 'fitness_level']


In [42]:
# Pipeline Stage 
numerical_transformer = Pipeline(steps =[
    ("scaler", MinMaxScaler())
])
categorical_transformer = Pipeline(steps = [
    ("encoder", OrdinalEncoder(handle_unknown = "use_encoded_value", unknown_value = -1))
])

In [43]:
# Column Transformer Stage
transformers = ColumnTransformer(
    transformers =[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [54]:
lr = LinearRegression()
dt = DecisionTreeRegressor()
svr = SVR()    # Soft Voting uchun
xgb = XGBRegressor(
    n_estimators = 200,
    learning_rate = 0.05,
    max_depth = 6,
    subsample_bytree = 0.8,
    n_jobs = -1,
    random_state = 42
)

In [55]:
#  Voting
voting_regressor = VotingRegressor(
    estimators= [("lr", lr), ("dt", dt), ("svr", svr), ("xgb", xgb)],
    # voting = "hard". sababi regressorda hard voting argument 
    n_jobs = -1
)

pipeline = Pipeline(steps = [
    ("preprocessor", transformers),
    ("model", voting_regressor)
])
pipeline.fit(X_train, y_train)
y_pred_hard = pipeline.predict(X_test)
print("Hard Voting R2_score: ", r2_score(y_test, y_pred_hard))
print("Hard Voting MSE: ", mean_squared_error(y_test, y_pred_hard))

/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [19:47:16] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "subsample_bytree" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Hard Voting R2_score:  0.9442587388038493
Hard Voting MSE:  24696.074911127893


In [50]:
# Joblib saving 
dump(pipeline, "voting_regressor.joblib")

['voting_regressor.joblib']

# Stacking 

In [56]:
base_learners = [
    ("lr", LinearRegression()),
    ("dt", DecisionTreeRegressor()),
    ("xgb", XGBRegressor(
        n_estimators = 200,
        learning_reate = 0.05,
        max_depth = 6,
        subsample_bytree = 0.8,
        n_jobs =-1,
        random_state = 42
    ))
]


In [58]:
# Level 1 Meta model 
meta_model = LinearRegression()
stack = StackingRegressor(estimators = base_learners, final_estimator = meta_model)

stack_pipeline = Pipeline(steps =[
    ("preprocessor", transformers),
    ("model", stack)
])
stack_pipeline.fit(X_train, y_train)
y_pred_stack = stack_pipeline.predict(X_test)
print("Stacking Regressor R2_score: ", r2_score(y_test, y_pred_stack))
print("Stacking Regressor MSE: ", mean_squared_error(y_test, y_pred_stack))

/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [19:59:12] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "learning_reate", "subsample_bytree" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [19:59:13] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "learning_reate", "subsample_bytree" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [19:59:13] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "learning_reate", "subsample_bytree" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/murodjongafforov/Desktop

Stacking Regressor R2_score:  1.0
Stacking Regressor MSE:  6.5478077096630024e-24


In [62]:
# base learners
base_learners = [
    ("lr", Pipeline([
        ("preprocessor", transformers), ("model", lr)
    ])),
    ("dt", Pipeline([
        ("preprocessor", transformers), ("model", dt)
    ])),
    ("xgb", Pipeline([
            ("preprocessor", transformers), ("model", xgb)
    ]))
]


# Predictions 
base_train_preds = []
base_test_preds = []

print("=== Base Models ===")

=== Base Models ===


In [63]:
for name, model in base_learners:
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    base_train_preds.append(y_train_pred)
    base_test_preds.append(y_test_pred)

    # Individually
    r2_train = model.score(X_train, y_train)
    r2_test = model.score(X_test, y_test)
    print(f"{name} - R2 train: {r2_train:.3f}, test: {r2_test:.3f}")
    

lr - R2 train: 1.000, test: 1.000
dt - R2 train: 1.000, test: 1.000


/Users/murodjongafforov/Desktop/ensemble_metadata/ensemble_metadata/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [20:15:17] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "subsample_bytree" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


xgb - R2 train: 1.000, test: 1.000


In [68]:
import numpy as np 

X_meta_train = np.column_stack(base_train_preds)
X_meta_test = np.column_stack(base_test_preds)

print(X_meta_train)
print(X_meta_test)

[[ 873.          873.          873.03466797]
 [2595.         2595.         2590.99145508]
 [1587.         1587.         1585.12939453]
 ...
 [ 750.          750.          750.16296387]
 [2401.         2401.         2401.62719727]
 [2347.         2347.         2351.77490234]]
[[1407.         1407.         1402.21887207]
 [ 750.          750.          750.12255859]
 [1242.         1242.         1243.31774902]
 ...
 [2378.         2377.         2370.41381836]
 [1641.         1641.         1643.6965332 ]
 [2050.         2050.         2048.1730957 ]]


In [69]:
meta_model = Ridge()
meta_model.fit(X_meta_train, y_train)

y_final_pred = meta_model.predict(X_meta_test)
r2_final = meta_model.score(X_meta_test, y_test)

print("=== Meta Model (stacking) ===")
print(f"Final Predictions: {y_final_pred[:5]}")
print(f"R2 test (final output): {r2_final:.3f}")

=== Meta Model (stacking) ===
Final Predictions: [1406.99993932  750.00000066 1242.00001632 1240.49997647 1694.50001653]
R2 test (final output): 1.000


In [70]:
# Saving Stacking 
dump(meta_model, "stacking_ridge.joblib")

['stacking_ridge.joblib']

# Bagging 

In [71]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor


In [77]:
# Models about bagging 

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Bagging": BaggingRegressor(
        estimator = DecisionTreeRegressor(),
        n_estimators = 100,
        random_state=42
    ),
    "Random Forest":RandomForestRegressor(
        n_estimators = 100,
        random_state = 42
    )
}

bagging_pipelines = {

    name: Pipeline(steps=[
    ("preprocessor", transformers),
    ("model", model)
])
for name, model in models.items()
}
# Train and Evaluation 
for name, bagging_pipeline in bagging_pipelines.items():
    bagging_pipeline.fit(X_train, y_train)
    y_pred_bag = bagging_pipeline.predict(X_test)

    mse = mean_squared_error(y_test, y_pred_bag)
    r2 = r2_score(y_test, y_pred_bag)
    print(f"{name}")
    print(f"R2 score: {r2:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"-" * 30)



Decision Tree
R2 score: 1.0000
MSE: 2.5933
------------------------------
Bagging
R2 score: 1.0000
MSE: 0.7120
------------------------------
Random Forest
R2 score: 1.0000
MSE: 0.6676
------------------------------


In [79]:
# Saving Stacking 
dump(bagging_pipeline, "bagging.joblib")

['bagging.joblib']

# Comparisons of all models

In [ ]:
from tabulate import tabulate


tabulat_data = [
    [
        "Complexity (Time)", 
        "2.5s, 1.8 MB on disk",   # Voting
        "3.3s, 4 KB on disk",  # Stacking 
        "3.8s, 23.8 MB on disk" # Bagging 
    ],
    [
        "Learning Combination", 
        "No", 
        "Yes", 
        "No"
    ],
    [
        "Risk of Overfitting", 
         "Low", 
        "High", 
        "High"
    ],
    [
        "Performance", 
        "Quite Good", 
        "Excellent", 
        "Excellent"
    ]
]

# Column names
headers = ["Aspect", "Voting Classifier", "Stacking Classifier", "Bagging"]

print(tabulate(tabulat_data, headers=headers, tablefmt="grid"))

+----------------------+----------------------+-----------------------+-----------------------+
| Aspect               | Voting Classifier    | Stacking Classifier   | Bagging               |
+======================+======================+=======================+=======================+
| Complexity (Time)    | 2.5s, 1.8 MB on disk | 3.3s, 4 KB on disk    | 3.8s, 23.8 MB on disk |
+----------------------+----------------------+-----------------------+-----------------------+
| Learning Combination | No                   | Yes                   | No                    |
+----------------------+----------------------+-----------------------+-----------------------+
| Risk of Overfitting  | Low                  | High                  | High                  |
+----------------------+----------------------+-----------------------+-----------------------+
| Performance          | Quite Good           | Excellent             | Excellent             |
+----------------------+----------------